In [1]:
import cadquery as cq
import math

In [2]:
# ----------------------------
# Parameters
# ----------------------------
output_file = "snake_scales_jnb.stl"

num_scales = 7

# Base plate
plate_length = 600.0  # along Y
plate_width  = 100.0  # along X
plate_height = 50.0

# Scale parameters
base_length = 70.0 # along Y, reduces/increases empty space between the scales
base_width = 60.0 # along X, increases/reduces empty space between base and tip
tip_width_lower = 10.0 # along Z, makes the tip thicker/spikier
tip_width_upper = 10.0 # along Y, makes the tip less/more triangular
path_length = plate_width - plate_width/4
attack_angle = 30.0
num_sections = 4
curvature_factor = 1.3

# Margins
margin_outer = 10.0 
margin_inner = 10.0 
x_offset_scale = 50.0

#create_scale_set()

# TODO: make the lower curve straight for printability
# TODO: vary base width/tip width ratio, height, angle of attack, base length and print with random numbers

In [3]:
def lerp(a, b, t):
    return a + (b - a) * t

def create_scale(base_width, base_length, tip_width_lower, tip_width_upper,
                 path_length, attack_angle, num_sections, curvature_factor):

    angle_rad = math.radians(attack_angle)
    
    y_end = path_length * math.cos(angle_rad)
    z_end = path_length * math.sin(angle_rad)
    
    y_start = base_length / 2
    
    spine = (
        cq.Workplane("YZ")
        .moveTo(y_start, 0)
        .threePointArc(
            (y_start + y_end * 0.5, z_end * curvature_factor),
            (y_start + y_end, z_end)
        )
        .val()
    )
    
    profiles = []
    
    Z = cq.Vector(0, 0, 1)
    Y = cq.Vector(0, 1, 0)
    X = cq.Vector(1, 0, 0)

    # Base trapezoid top edge
    base_top_length = base_length * 0.6
    
    # Generate trapezoid profiles
    for i in range(num_sections):
        t = i / (num_sections - 1)
    
        point = spine.positionAt(t)
    
        t_rot = t**1.5
        normal = (Z.multiply(1 - t_rot) + Y.multiply(t_rot)).normalized()
    
        x_dir = X
        y_dir = normal.cross(x_dir).normalized()
    
        plane = cq.Plane(origin=point, xDir=x_dir, normal=normal)
    
        # Trapezoid height (Y direction)
        width = lerp(base_width, tip_width_upper, t)
    
        # Bottom edge (X direction, lower side)
        bottom_len = lerp(base_length, tip_width_lower, t)
    
        # Top edge (X direction, upper side)
        top_len = lerp(base_top_length, tip_width_upper, t)
    
        pts = [
            (-bottom_len/2, -width/2),
            ( bottom_len/2, -width/2),
            ( top_len/2,    width/2),
            (-top_len/2,    width/2)
        ]
    
        wire = cq.Workplane(plane).polyline(pts).close().val()
        profiles.append(wire)
    
    scale = cq.Solid.makeLoft(profiles, ruled=False)
    return scale

In [4]:
def create_scale_set():
    
    # ----------------------------
    # Create base plate
    # ----------------------------
    plate = cq.Workplane("XY").rect(plate_width, plate_length).extrude(plate_height)
    
    # ----------------------------
    # Compute spacing along Y
    # ----------------------------
    available_length = plate_length - margin_outer - margin_inner - num_scales * base_length
    if num_scales > 1:
        spacing = available_length / (num_scales - 1)
    else:
        spacing = 0.0
    
    if spacing < 0:
        raise ValueError("Scales do not fit on plate. Increase plate_length or reduce scale size/number.")
    
    # ----------------------------
    # Place scales along Y
    # ----------------------------
    scales = cq.Workplane("XY")
    y = -plate_length / 2 + margin_outer + base_length / 2
    
    for i in range(num_scales):
        scale = create_scale(
            base_width,          # trapezoid height at base
            base_length,         # bottom edge at base
            tip_width_lower,     # bottom edge at tip
            tip_width_upper,     # top edge + height driver at tip
            path_length,
            attack_angle,
            num_sections,
            curvature_factor
        )
    
        scale = scale.rotate((0,0,0), (0,0,1), 90)
        scale = scale.translate((x_offset_scale, y, plate_height))
    
        scales = scales.add(scale)
    
        y += base_length + spacing
    
    # ----------------------------
    # Combine scales and plate
    # ----------------------------
    result = plate.union(scales)
    display(result)

In [5]:
create_scale_set()